In [1]:
import tensorflow as tf

import pandas as pd
import numpy as np
import os, pickle


from network import  NetCNN2D_CSP, build_functional_cnn2D
from mne.decoding import CSP
from dataset_NewEEG import filter_rawEEG

from config_NewEEG import Config
import dataset_NewEEG


from quantizeml.models import QuantizationParams, quantize


import requests



server_ip =  "http://192.168.216.53:8000"




csv_path = os.path.join("streaming_data_file", "Conscious_EEG_20241203_152326063.csv")
data_raw = pd.read_csv(csv_path, skiprows=2, delimiter=';')
data_raw = data_raw[['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'C1', 'Cz', 'C2', 'P3', 'Pz', 'P4', 'T3', 'T4']]

def get_data_epoch():
    return np.array(data_raw.loc[0:2*250 - 1]).transpose(1,0)



from keras.utils import get_custom_objects

get_custom_objects()['NetCNN2D_CSP'] = NetCNN2D_CSP



model = tf.keras.models.load_model(os.path.join('saved_models', 'best_model.tf'))

with open(os.path.join('saved_models', 'csp.pkl'), 'rb') as f:
    csp_trained = pickle.load(f)



In [2]:
functional_model = build_functional_cnn2D(n_classes=2, input_shape=(3, 500, 1))
functional_model.set_weights(model.get_weights())


config = Config()

config.data_path = "EEG_RAW_DATA"
config.test_session = 1
config.used_classes = [['CLeft'], ['CDown']]

X, y = dataset_NewEEG.session_dataset(config, f'I{config.test_session:02d}')
X, _ = dataset_NewEEG.slice_EEG_epoch(config, X, y)
X = csp_trained.transform(X)
X = np.expand_dims(X, 3)
Xq = ((X - X.min()) / (X.max() - X.min()) * 255).astype(np.uint8)

qparams = QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4, per_tensor_activations=True)
quantized_model = quantize(functional_model, qparams=qparams, samples=Xq, num_samples=120, batch_size=16, epochs=5)
quantized_model.summary()

8/8 [==============================] - 0s 3ms/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 3, 500, 1)]       0         
                                                                 
 conv2d_2 (QuantizedConv2D)  (None, 3, 72, 8)          272       
                                                                 
 re_lu_3 (QuantizedReLU)     (None, 3, 72, 8)          2         
                                                                 
 conv2d_3 (QuantizedConv2D)  (None, 3, 15, 16)         2704      
                                                                 
 re_lu_4 (QuantizedReLU)     (None, 3, 15, 16)         2         
                                                                 
 flatten_1 (QuantizedFlatte  (None, 720)               0         
 n)                                                              
          

In [ ]:
import akida
from cnn2snn import convert, set_akida_version, AkidaVersion


#virtual_chip = akida.devices()[0] #akida.AKD1000()
chip = akida.devices()[0]
print(chip)

with set_akida_version(AkidaVersion.v1):
    model_akida = convert(quantized_model)
    model_akida.summary()


#model_akida.map(virtual_chip)
model.akida.map(chip)

y = model_akida.forward(Xq)

                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[3, 500, 1]  [1, 1, 2]     1          4     
______________________________________________

_______________________________________________________
Layer (type)           Output shape  Kernel shape    

============ SW/conv2d_2-dense_3 (Software) ===========

conv2d_2 (InputConv.)  [3, 72, 8]    (3, 11, 1, 8)   
_______________________________________________________
conv2d_3 (Conv.)       [3, 15, 16]   (3, 7, 8, 16)   
_______________________________________________________
dense_2 (Fully.)       [1, 1, 128]   (1, 1, 720, 128)
_______________________________________________________
dense_3 (Fully.)       [1, 1, 2]     (1, 1, 128, 2)  
_______________________________________________________


In [ ]:
from Serial_class import serial_class
import threading
import time
import sys

if __name__ == '__main__':

    #Gestion port com
    serial_flux = serial_class()
    serial_flux.init_port()

    thread_a = threading.Thread(target=serial_flux.reception, name='ta')
    thread_a.start()

    time.sleep(2)
    cptEEG =0

    if not serial_flux.data:
        print('No serial flux , program is closing')
        serial_flux.terminate()
        sys.exit()


    EEGraw_stack = []

    try:
        while True:
            while not serial_flux.data_queue.empty():
                array_data_list , ts_data = serial_flux.data_queue.get()
                #print(len(array_data_list))
                #print(array_data_list[0])
                #time.sleep(1)
                for i in range(len(array_data_list)):
                    ts_value = ts_data[i] + (i*4) # Obtient le timestamp associe
                    EEGraw_stack.append(array_data_list[i][:13])
                    cptEEG +=1


                if len(EEGraw_stack) > 500:
                    last_epoch_raw = EEGraw_stack[-500:]
                    X = np.array(last_epoch_raw)
                    X = X.transpose(1,0)


                    if len(EEGraw_stack) > 10000:
                        EEGraw_stack[:] = last_epoch_raw

                    

                    X = filter_rawEEG(X, 0.5, 35)
                    X = np.expand_dims(X, 0)

                    X = csp_trained.transform(X)
                    X = np.expand_dims(X, 3)

                    # Linear Quantization
                    Xq = ((X - X.min()) / (X.max() - X.min()) * 255).astype(np.uint8)

                    out = np.argmax(model_akida.forward(Xq))
                    

                    direction = "left" if out == 0 else "right"
                    print(direction)

                    requests.post(f"{server_ip}/push_direction", json={"direction": direction})



            
            time.sleep(0.002)  # Ajouter un delai pour eviter d'occuper 100 du CPU
            if (cptEEG >250 *30):
                serial_flux.terminate() # Fermeture de la connexion EEG
                thread_a.join()
                break

    except KeyboardInterrupt:
        # Arreter le thread proprement lors d'une interruption (Ctrl + C)
        serial_flux.terminate() # Fermeture de la connexion EEG
        thread_a.join()


Port: COM3
  Description : Silicon Labs CP210x USB to UART Bridge (COM3)
  Hardware ID : USB VID:PID=10C4:EA60 SER=0289339D LOCATION=1-2
  Vendor ID   : 4292
  Product ID  : 60000
  Manufacturer: Silicon Labs
  Serial Num  : 0289339D
  Location    : 1-2
  Product     : None

port : COM3
right
left
left
left
left
left
left
left
right
left
left
left
left
left
left
left
left
left
left
left
left
left
left
right
left
left
left
left
left
left
left
left
left
left
left
left
left
right
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
left
right


In [12]:
import requests

server_ip =  "http://192.168.216.53:8000"

while True:
    raw_epoch_data = get_data_epoch()
    print(raw_epoch_data.shape)
    X = filter_rawEEG(raw_epoch_data, 0.5, 35)
    X = np.expand_dims(X, 0)

    X = csp_trained.transform(X)
    X = np.expand_dims(X, 3)

    # Linear Quantization
    Xq = ((X - X.min()) / (X.max() - X.min()) * 255).astype(np.uint8)

    out = np.argmax(model_akida.forward(Xq))
    

    direction = "left" if out == 0 else "right"
    print(direction)

    #requests.post(f"{server_ip}/push_direction", json={"direction": direction})



(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)
left
(13, 500)


KeyboardInterrupt: 